# Model Evaluation & Optimization

Avalia os modelos segundo as métricas relevantes para deteção de malware. A comparação e a afinação usam apenas treino; a validação escolhe o limiar para a classe maliciosa; o teste fica reservado para a avaliação final.

In [11]:
from datetime import datetime
from pathlib import Path
import sys

import pandas as pd

SRC_PATH = Path("..").resolve() / "src"
sys.path.insert(0, str(SRC_PATH))

import data_preparation as dp
import data_modelling as dm
import data_evaluation as de


## 1. Preparar os dados

Reproduz a divisão estratificada definida em Data Preparation. Neste projeto, `0` corresponde a benigno e `1` a malicioso.

In [12]:
DATASET_NAME = "malware_memory_dump"
DATA_PATH = Path("..").resolve() / "dataset" / "raw" / "MalwareMemoryDump.csv"
OUTPUT_DIR = Path("..").resolve() / "models"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
POSITIVE_LABEL = 1

X_train, X_val, X_test, y_train, y_val, y_test = dp.run_data_preparation(
    str(DATA_PATH),
    target="Label",
    random_state=42,
)


Dimensões do dataset: (58596, 58)

--- Valores nulos ---
Raw_Type                                  0
pslist_nproc                              0
pslist_nppid                              0
pslist_avg_threads                        0
pslist_nprocs64bit                        0
pslist_avg_handlers                       0
dlllist_ndlls                             0
dlllist_avg_dlls_per_proc                 0
handles_nhandles                          0
handles_avg_handles_per_proc              0
handles_nport                             0
handles_nfile                             0
handles_nevent                            0
handles_ndesktop                          0
handles_nkey                              0
handles_nthread                           0
handles_ndirectory                        0
handles_nsemaphore                        0
handles_ntimer                            0
handles_nsection                          0
handles_nmutant                           0
ldrmodules_not_in_l

## 2. Selecionar e afinar o modelo

A comparação usa F1 macro em validação cruzada estratificada. A pesquisa de hiperparâmetros do modelo selecionado também usa apenas treino; a validação não participa nesta escolha.

In [13]:
results_df = dm.evaluate_models(X_train, y_train, scoring="f1_macro")
dm.save_results_csv(
    results_df,
    dataset_name=DATASET_NAME,
    output_dir=str(OUTPUT_DIR),
    timestamp=RUN_TIMESTAMP,
)


Regressão Logística    | f1_macro = 0.9989 ± 0.0003
kNN                    | f1_macro = 0.9998 ± 0.0001
SVM                    | f1_macro = 0.9996 ± 0.0002
Naive Bayes            | f1_macro = 0.9927 ± 0.0007
MLP                    | f1_macro = 0.9999 ± 0.0001
Árvore de Decisão      | f1_macro = 0.9997 ± 0.0002
Random Forest          | f1_macro = 0.9999 ± 0.0001
Gradient Boosting      | f1_macro = 0.9996 ± 0.0002
Comparação de modelos guardada em: C:\Users\Thiago\Desktop\IAAC\models\comparacao_modelos_malware_memory_dump_20261007_165810_606908.csv


'C:\\Users\\Thiago\\Desktop\\IAAC\\models\\comparacao_modelos_malware_memory_dump_20261007_165810_606908.csv'

In [14]:
best_name = dm.select_best_model(results_df)
search = de.tune_hyperparameters(
    best_name,
    X_train,
    y_train,
    scoring="f1_macro",
    n_splits=5,
    random_state=42,
)
best_pipe = search.best_estimator_



Melhor modelo: Random Forest
Melhores hiperparametros por CV (Random Forest): {'model__max_depth': None, 'model__n_estimators': 300}
f1_macro medio na CV: 1.0000


## 3. Escolher o limiar na validação

O limiar maximiza o F1 da classe maliciosa na validação. O modelo permanece ajustado apenas em treino para que a validação continue independente do ajuste do estimador. O resultado deve ser interpretado em conjunto com recall e falsos negativos.

In [15]:
val_scores, score_type = de.get_positive_class_scores(
    best_pipe,
    X_val,
    positive_label=POSITIVE_LABEL,
)
threshold_result = de.optimize_threshold(
    y_val,
    val_scores,
    positive_label=POSITIVE_LABEL,
)
selected_threshold = threshold_result["selected"]["threshold"]


Limiar escolhido na validacao: 0.713333 | F1 malware=1.0000 | recall malware=1.0000


## 4. Avaliação final no teste

O conjunto de teste é usado depois de fixados o modelo, hiperparâmetros e limiar. São reportados o limiar padrão e o limiar escolhido na validação; só o segundo é a configuração otimizada.

In [16]:
test_results = de.evaluate_test_thresholds(
    best_pipe,
    X_test,
    y_test,
    selected_threshold,
    score_type,
    positive_label=POSITIVE_LABEL,
    dataset_name=DATASET_NAME,
    output_dir=str(OUTPUT_DIR),
    timestamp=RUN_TIMESTAMP,
)


## 5. Guardar avaliação e modelo

Os artefactos ficam em `models/` com timestamp. O pipeline guardado permanece ajustado apenas no treino, compatível com o limiar escolhido na validação; o JSON inclui parâmetros, métricas e limiar para utilização posterior.

In [17]:
evaluation_metrics = {
    "dataset": DATASET_NAME,
    "model": best_name,
    "positive_class": "Malicioso (1)",
    "n_splits": 5,
    "cv_best_f1_macro": float(search.best_score_),
    "best_params": search.best_params_,
    "threshold_selection": {
        **threshold_result["selected"],
        "score_type": score_type,
        "objective": "maximizar F1 da classe maliciosa na validacao",
    },
    "test": test_results,
}
artifact_paths = de.save_evaluation_artifacts(
    evaluation_metrics,
    threshold_result["curve"],
    dataset_name=DATASET_NAME,
    output_dir=str(OUTPUT_DIR),
    timestamp=RUN_TIMESTAMP,
)
model_path = dm.save_model(
    best_pipe,
    best_name,
    dataset_name=DATASET_NAME,
    output_dir=str(OUTPUT_DIR),
    metrics=evaluation_metrics,
    feature_names=list(X_train.columns),
    timestamp=RUN_TIMESTAMP,
)
print(f"Modelo guardado em: {model_path}")
artifact_paths


Curva de limiar guardada em: C:\Users\Thiago\Desktop\IAAC\models\curva_limiar_malware_memory_dump_20261007_165810_606908.csv
Grafico de limiar guardado em: C:\Users\Thiago\Desktop\IAAC\models\otimizacao_limiar_malware_memory_dump_20261007_165810_606908.png
Relatorio de avaliacao guardado em: C:\Users\Thiago\Desktop\IAAC\models\avaliacao_malware_memory_dump_20261007_165810_606908.json
Modelo guardado em: C:\Users\Thiago\Desktop\IAAC\models\malware_memory_dump_random_forest_20261007_165810_606908.joblib
Metadados guardados em: C:\Users\Thiago\Desktop\IAAC\models\malware_memory_dump_random_forest_20261007_165810_606908.json
Cópias 'latest' atualizadas em: C:\Users\Thiago\Desktop\IAAC\models\malware_memory_dump_random_forest_latest.joblib / C:\Users\Thiago\Desktop\IAAC\models\malware_memory_dump_random_forest_latest.json
Modelo guardado em: C:\Users\Thiago\Desktop\IAAC\models\malware_memory_dump_random_forest_20261007_165810_606908.joblib


{'threshold_curve': 'C:\\Users\\Thiago\\Desktop\\IAAC\\models\\curva_limiar_malware_memory_dump_20261007_165810_606908.csv',
 'threshold_plot': 'C:\\Users\\Thiago\\Desktop\\IAAC\\models\\otimizacao_limiar_malware_memory_dump_20261007_165810_606908.png',
 'report': 'C:\\Users\\Thiago\\Desktop\\IAAC\\models\\avaliacao_malware_memory_dump_20261007_165810_606908.json'}